Module Description:
-------------------
Class to extract skills from text and align them to existing taxonomy

Ownership:
----------
Project: Leveraging Artificial intelligence for Skills Extraction and Research (LAiSER)
Owner:  George Washington University Institute of Public Policy
        Program on Skills, Credentials and Workforce Policy
        Media and Public Affairs Building
        805 21st Street NW
        Washington, DC 20052
        PSCWP@gwu.edu
        https://gwipp.gwu.edu/program-skills-credentials-workforce-policy-pscwp

License:
--------
Copyright 2024 George Washington University Institute of Public Policy

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated
documentation files (the “Software”), to deal in the Software without restriction, including without limitation
the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software,
and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the
Software.

THE SOFTWARE IS PROVIDED “AS IS”, WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE
WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR
COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR
OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

Revision History:
-----------------
| Rev No. | Date | Author | Description |
|---------|------|--------|-------------|
| [1.0.0] | 06/05/2024 |      Satya Phanindra K. |  Created a standalone notebook for skill extraction
| [1.0.1] | 06/11/2024 |      Satya Phanindra K. |  Added GPU support for processing
| [1.0.1] | 06/20/2024 |      Satya Phanindra K. |  Added error handling and logging
| [1.0.2] | 07/01/2024 |      Satya Phanindra K. |  Threshold update for similarity and AI model
| [1.0.3] | 07/10/2024 |      Satya Phanindra K. |  Added seperate functions set for LLM usecases
| [1.0.4] | 07/13/2024 |      Satya Phanindra K. |  Add descriptions to each method
| [1.0.5] | 07/18/2024 |      Satya Phanindra K. |  Added CONDITIONAL GPU support for LLM
| [1.0.6] | 07/22/2024 |      Satya Phanindra K. |  Added support for SkillNer model for skill extraction, if GPU not available
| [1.0.7] | 07/25/2024 |      Satya Phanindra K. |  Calculate cosine similarities in bulk for optimal performance.
| [1.0.8] | 07/28/2024 |      Satya Phanindra K. |  Error handling for empty list outputs from extract_raw function
| [1.0.9] | 11/24/2024 |      Prudhvi Chekuri    |  Add functionality to extract skills from syllabi data.
| [1.1.0] | 03/14/2025 |      Deepika Reddygari  |  Import laiser as a python package.
| [1.1.1] | 03/15/2025 |      Bharat Khandelwal  |  Resolved all issues related to importing laiser as a python package.
| [1.1.2] | 03/19/2025 |      Satya Phanindra K. |  Update installation with uv.
| [1.1.3] | 04/02/2025 |      Prudhvi Chekuri    |  Fix dependency issues.

## Install and import LAiSER

Switching to installation with pip until the errors in uv package are fixed.

In [1]:
#!pip install uv
#!uv pip install dev-laiser -q
!pip install dev-laiser -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 8.7 MB/s eta 0:00:00
   

**NOTE**: If running on Google Colab, RESTART the runtime for a True Clean Slate before executing below code. (**REQUIRED**)

In [2]:
from laiser.skill_extractor import Skill_Extractor
import pandas as pd
import torch

INFO 04-14 10:16:20 [__init__.py:239] Automatically detected platform cuda.


## Using the Skill Extractor

#### With Job Descriptions

In [3]:
# Import the dataset
job_sample = pd.read_csv('https://raw.githubusercontent.com/rafal-potentially/skills-extraction/refs/heads/main/sample.csv')

job_sample = job_sample[0:1]
job_sample = job_sample[['description', 'job_id']]
print("Considering", len(job_sample), "rows for processing...")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 42, saw 47


In [ ]:
job_sample

In [ ]:
print('Initializing the Skill Extractor...')
se = Skill_Extractor(AI_MODEL_ID="marcsun13/gemma-2-9b-it-GPTQ", HF_TOKEN="<YOUR_HUGGING_FACE_API_TOKEN>", use_gpu=True)
print('The Skill Extractor has been initialized successfully!')

In [ ]:
# skills output based on the taxonomy database
output = se.extractor(job_sample, 'job_id', text_columns = ['description'])

In [ ]:
# save the extracted skills to a csv file
display(output)
output.to_csv('extracted_skills_for_sample_jobs.csv', index=False)

#### With syllabi

In [ ]:
syllabi_sample = pd.read_csv("https://raw.githubusercontent.com/LAiSER-Software/datasets/refs/heads/master/syllabi-data/preprocessed_50_opensyllabus_syllabi_data.csv")
syllabi_sample = syllabi_sample[0:1]
syllabi_sample = syllabi_sample[['id', 'description', 'learning_outcomes']]
print("Considering", len(syllabi_sample), "rows for processing...")

In [ ]:
syllabi_sample

In [ ]:
output = se.extractor(syllabi_sample, 'id', text_columns = ['description', 'learning_outcomes'], input_type = "syllabus")

In [ ]:
# save the extracted skills to a csv file
display(output)
output.to_csv('extracted_skills_for_sample_syllabus.csv', index=False)

None of the raw skills extracted from this sample has high correlation with the taxonomy skills. Considering a different sample...

In [ ]:
syllabi = pd.read_csv("https://raw.githubusercontent.com/LAiSER-Software/datasets/refs/heads/master/syllabi-data/preprocessed_50_opensyllabus_syllabi_data.csv")
syllabi = syllabi[['id', 'description', 'learning_outcomes']]
syllabi_sample = syllabi[1:2]
print("Considering", len(syllabi_sample), "rows for processing...")

output = se.extractor(syllabi_sample, 'id', text_columns = ['description', 'learning_outcomes'], input_type = "syllabus")

# save the extracted skills to a csv file
display(output)
output.to_csv('extracted_skills_for_sample_syllabus.csv', index=False)